In [ ]:
#LIBRERIAS

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import skew, kurtosis, zscore
from wordcloud import WordCloud
import warnings
warnings.filterwarnings('ignore')
import unicodedata
from collections import Counter
import os


In [ ]:
# Configuración profesional de visualizaciones
plt.rcParams.update({
    'figure.figsize': (16, 10),
    'font.size': 10,
    'axes.titlesize': 12,
    'axes.labelsize': 11,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
    'legend.fontsize': 10,
    'figure.dpi': 100,
    'savefig.dpi': 100,
})

sns.set_style("whitegrid")
sns.set_palette("husl")

print("Librerías importadas correctamente\n")

✅ Librerías importadas correctamente



In [ ]:
def cargar_datasets():
    """
    Carga los 3 datasets con detección automática de encoding
    Retorna diccionario con los 3 dataframes
    """
    from google.colab import files

    print("\nSelecciona los archivos a cargar...")
    print("   Debes subir: prendas_base_500_mejorado.csv, outfits_100k_completo.csv, test_30_predefinido.csv")

    uploaded = files.upload()

    datasets = {}

    for filename in uploaded.keys():
        print(f"\nCargando {filename}...")
        try:
            df = pd.read_csv(filename, encoding='utf-8')
            datasets[filename.replace('.csv', '')] = df
            print(f"Cargado: {df.shape[0]:,} filas × {df.shape[1]} columnas")
        except Exception as e:
            print(f"Error: {e}")

    return datasets

# Cargar datasets
print("\nIniciando carga de archivos...")
datasets = cargar_datasets()

print(f"\nTotal datasets cargados: {len(datasets)}")
print(f" {list(datasets.keys())}\n")


Iniciando carga de archivos...

Selecciona los archivos a cargar...
   Debes subir: prendas_base_500_mejorado.csv, outfits_100k_completo.csv, test_30_predefinido.csv


Saving test_30_predefinido.csv to test_30_predefinido.csv
Saving outfits_100k_completo.csv to outfits_100k_completo.csv
Saving prendas_base_500_mejorado.csv to prendas_base_500_mejorado.csv

📖 Cargando test_30_predefinido.csv...
Cargado: 30 filas × 7 columnas

📖 Cargando outfits_100k_completo.csv...
Cargado: 100,000 filas × 6 columnas

📖 Cargando prendas_base_500_mejorado.csv...
Cargado: 500 filas × 1 columnas

Total datasets cargados: 3
 ['test_30_predefinido', 'outfits_100k_completo', 'prendas_base_500_mejorado']



In [ ]:
#FUNCIONES REUTILIZABLES

class DataScientistAnalyzer:
    """Clase profesional para análisis EDA avanzado"""

    def __init__(self, name, df):
        self.name = name
        self.df = df.copy()
        self.df_original = df.copy()
        self.report = {}

    def normalizar_utf8(self, texto):
        """Normaliza UTF-8 a forma compuesta NFC"""
        if pd.isna(texto):
            return texto
        return unicodedata.normalize('NFC', str(texto))

    def info_general(self):
        """Información general del dataset"""
        print(f"INFORMACIÓN GENERAL - {self.name.upper()}")

        print(f"\nShape: {self.df.shape[0]:,} filas × {self.df.shape[1]} columnas")
        print(f"Memoria: {self.df.memory_usage(deep=True).sum() / 1024 / 1024:.2f} MB")
        print(f"Tipos de datos:\n{self.df.dtypes}")

        self.report['shape'] = self.df.shape
        self.report['memory_mb'] = self.df.memory_usage(deep=True).sum() / 1024 / 1024

    def mostrar_columnas(self):
        """Mostrar todas las columnas con descripción"""
        print(f"NOMBRES DE COLUMNAS - {self.name.upper()}")


        print(f"\nTotal de columnas: {len(self.df.columns)}\n")
        for i, col in enumerate(self.df.columns, 1):
            dtype = self.df[col].dtype
            nunique = self.df[col].nunique()
            nulos = self.df[col].isnull().sum()

            print(f"{i:2d}. {col:25s} | Tipo: {str(dtype):12s} | Únicos: {nunique:6d} | Nulos: {nulos:5d}")

    def estadisticas_descriptivas(self):
        """Estadísticas descriptivas profundas"""
        print(f"ESTADÍSTICAS DESCRIPTIVAS - {self.name.upper()}")
        # Columnas numéricas
        numeric_cols = self.df.select_dtypes(include=[np.number]).columns

        if len(numeric_cols) > 0:
            print(f"\nVARIABLES NUMÉRICAS ({len(numeric_cols)}):")
            stats_df = self.df[numeric_cols].describe().T
            stats_df['skewness'] = self.df[numeric_cols].apply(skew)
            stats_df['kurtosis'] = self.df[numeric_cols].apply(kurtosis)
            print(stats_df.round(3))

        # Columnas categóricas
        cat_cols = self.df.select_dtypes(include=['object']).columns

        if len(cat_cols) > 0:
            print(f"\nVARIABLES CATEGÓRICAS ({len(cat_cols)}):")
            for col in cat_cols:
                print(f"\n   {col}:")
                print(f"      Únicos: {self.df[col].nunique()}")
                print(f"      Modo: {self.df[col].mode()[0] if len(self.df[col].mode()) > 0 else 'N/A'}")
                print(f"      Frecuencia del modo: {self.df[col].value_counts().iloc[0] if len(self.df[col].value_counts()) > 0 else 'N/A'}")

    def analizar_datos_faltantes(self):
        """Análisis profundo de datos faltantes"""
        print(f"\n{'='*90}")
        print(f"ANÁLISIS DE DATOS FALTANTES - {self.name.upper()}")
        print(f"{'='*90}")

        missing = self.df.isnull().sum()
        missing_pct = (missing / len(self.df)) * 100
        missing_df = pd.DataFrame({
            'Columna': missing.index,
            'Cantidad': missing.values,
            'Porcentaje': missing_pct.values
        }).sort_values('Cantidad', ascending=False)

        total_nulos = missing.sum()
        print(f"\nTotal de valores nulos: {total_nulos:,}")
        print(f"Porcentaje total: {(total_nulos / (self.df.shape[0] * self.df.shape[1]) * 100):.2f}%")

        if total_nulos > 0:
            print(f"\nDetalles por columna:")
            print(missing_df[missing_df['Cantidad'] > 0].to_string(index=False))

            # Visualizar
            fig, ax = plt.subplots(figsize=(12, 6))
            sns.heatmap(self.df.isnull(), cbar=True, yticklabels=False, ax=ax, cmap='RdYlGn_r')
            ax.set_title(f'Heatmap de Datos Faltantes - {self.name}', fontsize=14, fontweight='bold')
            plt.tight_layout()
            plt.show()
        else:
            print("No hay valores nulos en el dataset")

        self.report['missing_data'] = total_nulos

    def detectar_outliers(self):
        """Detección de outliers (IQR + Z-score)"""
        print(f"DETECCIÓN DE OUTLIERS - {self.name.upper()}")

        numeric_cols = self.df.select_dtypes(include=[np.number]).columns

        if len(numeric_cols) == 0:
            print("No hay columnas numéricas para analizar outliers")
            return

        outliers_iqr = {}
        outliers_zscore = {}

        print(f"\nMÉTODO 1: IQR (Interquartile Range)")
        print(f" Fórmula: Q1 - 1.5×IQR, Q3 + 1.5×IQR")

        for col in numeric_cols:
            Q1 = self.df[col].quantile(0.25)
            Q3 = self.df[col].quantile(0.75)
            IQR = Q3 - Q1
            lower = Q1 - 1.5 * IQR
            upper = Q3 + 1.5 * IQR

            outliers = self.df[(self.df[col] < lower) | (self.df[col] > upper)]
            outliers_iqr[col] = len(outliers)

            if len(outliers) > 0:
                pct = (len(outliers) / len(self.df)) * 100
                print(f"   {col}: {len(outliers)} outliers ({pct:.2f}%) | Rango: [{lower:.2f}, {upper:.2f}]")

        print(f"\nMÉTODO 2: Z-Score")
        print(f"   Fórmula: |z| > 3 (muy extremo)")

        for col in numeric_cols:
            z_scores = np.abs(zscore(self.df[col].dropna()))
            outliers = np.sum(z_scores > 3)
            outliers_zscore[col] = outliers

            if outliers > 0:
                pct = (outliers / len(self.df)) * 100
                print(f"   {col}: {outliers} outliers ({pct:.2f}%)")

        # Visualizar outliers
        if len(numeric_cols) > 0:
            fig, axes = plt.subplots(2, 2, figsize=(14, 10))
            axes = axes.flatten()

            for idx, col in enumerate(numeric_cols[:4]):
                ax = axes[idx]
                self.df[col].plot(kind='box', ax=ax)
                ax.set_title(f'Box Plot - {col}', fontweight='bold')
                ax.set_ylabel(col)

            plt.tight_layout()
            plt.show()

        self.report['outliers_iqr'] = outliers_iqr
        self.report['outliers_zscore'] = outliers_zscore

    def analizar_correlaciones(self):
        """Análisis de correlaciones (si hay variables numéricas)"""
        print(f"ANÁLISIS DE CORRELACIONES - {self.name.upper()}")
        numeric_cols = self.df.select_dtypes(include=[np.number]).columns

        if len(numeric_cols) < 2:
            print("Se necesitan al menos 2 variables numéricas para correlaciones")
            return

        print(f"\nMatriz de correlación ({len(numeric_cols)} variables):")
        corr_matrix = self.df[numeric_cols].corr()
        print(corr_matrix.round(3))

        # Correlaciones significativas (>0.3)
        print(f"\nCorrelaciones significativas (|r| > 0.3):")
        significant = []
        for i in range(len(corr_matrix.columns)):
            for j in range(i+1, len(corr_matrix.columns)):
                if abs(corr_matrix.iloc[i, j]) > 0.3:
                    significant.append({
                        'Variable 1': corr_matrix.columns[i],
                        'Variable 2': corr_matrix.columns[j],
                        'Correlación': corr_matrix.iloc[i, j]
                    })

        if significant:
            sig_df = pd.DataFrame(significant).sort_values('Correlación', key=abs, ascending=False)
            print(sig_df.to_string(index=False))
        else:
            print("   No hay correlaciones significativas encontradas")

        # Heatmap
        if len(numeric_cols) > 1:
            fig, ax = plt.subplots(figsize=(12, 8))
            sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0,
                       square=True, ax=ax, cbar_kws={"shrink": 0.8})
            ax.set_title(f'Heatmap de Correlaciones - {self.name}', fontsize=14, fontweight='bold')
            plt.tight_layout()
            plt.show()

        self.report['correlations'] = corr_matrix

    def analizar_distribuciones(self):
        """Análisis profundo de distribuciones"""
        print(f"ANÁLISIS DE DISTRIBUCIONES - {self.name.upper()}")

        # Variables numéricas
        numeric_cols = self.df.select_dtypes(include=[np.number]).columns

        if len(numeric_cols) > 0:
            print(f"\nDISTRIBUCIONES DE VARIABLES NUMÉRICAS:")
            fig, axes = plt.subplots(2, 2, figsize=(14, 10))
            axes = axes.flatten()

            for idx, col in enumerate(numeric_cols[:4]):
                ax = axes[idx]
                self.df[col].hist(bins=30, ax=ax, edgecolor='black', alpha=0.7)
                ax.set_title(f'Distribución - {col}', fontweight='bold')
                ax.set_xlabel(col)
                ax.set_ylabel('Frecuencia')

                # Agregar estadísticas
                mean = self.df[col].mean()
                median = self.df[col].median()
                ax.axvline(mean, color='r', linestyle='--', label=f'Media: {mean:.2f}')
                ax.axvline(median, color='g', linestyle='--', label=f'Mediana: {median:.2f}')
                ax.legend()

            plt.tight_layout()
            plt.show()

        # Variables categóricas
        cat_cols = self.df.select_dtypes(include=['object']).columns

        if len(cat_cols) > 0:
            print(f"\nDISTRIBUCIONES DE VARIABLES CATEGÓRICAS:")
            fig, axes = plt.subplots(2, 2, figsize=(14, 10))
            axes = axes.flatten()

            for idx, col in enumerate(cat_cols[:4]):
                ax = axes[idx]
                value_counts = self.df[col].value_counts()
                if len(value_counts) > 10:
                    value_counts.head(10).plot(kind='barh', ax=ax, color='steelblue')
                    ax.set_title(f'{col} (Top 10)', fontweight='bold')
                else:
                    value_counts.plot(kind='barh', ax=ax, color='steelblue')
                    ax.set_title(f'{col}', fontweight='bold')
                ax.set_xlabel('Frecuencia')

            plt.tight_layout()
            plt.show()

    def generar_wordcloud(self):
        """Generar Word Cloud para palabras más frecuentes"""
        print(f"WORD CLOUD - PALABRAS MÁS FRECUENTES - {self.name.upper()}")


        text_cols = self.df.select_dtypes(include=['object']).columns

        if len(text_cols) == 0:
            print("No hay columnas de texto para generar word cloud")
            return

        # Combinar todo el texto
        all_text = ' '.join(self.df[text_cols].fillna('').values.flatten().astype(str))

        # Palabras individuales con frecuencia
        words = all_text.lower().split()
        word_freq = Counter(words)

        print(f"\nTop 30 palabras más frecuentes:")
        for word, freq in word_freq.most_common(30):
            if len(word) > 2:  # Ignorar palabras muy cortas
                print(f"   {word:25s}: {freq:5d} veces")

        # Generar Word Cloud
        if len(all_text) > 0:
            fig, ax = plt.subplots(figsize=(14, 8))
            wc = WordCloud(width=1000, height=600, background_color='white',
                          colormap='viridis', max_words=100,
                          relative_scaling=0.5, min_font_size=10).generate(all_text)
            ax.imshow(wc, interpolation='bilinear')
            ax.set_title(f'Word Cloud - {self.name}', fontsize=16, fontweight='bold', pad=20)
            ax.axis('off')
            plt.tight_layout()
            plt.show()

        self.report['word_freq'] = word_freq.most_common(50)

    def limpiar_datos(self):
        """Limpieza profunda de datos"""
        print(f"LIMPIEZA DE DATOS - {self.name.upper()}")
        n_antes = len(self.df)
        print(f"\nEstado inicial: {n_antes:,} registros")

        # 1. Normalizar UTF-8
        print(f"\nNormalizando UTF-8...")
        text_cols = self.df.select_dtypes(include=['object']).columns
        for col in text_cols:
            self.df[col] = self.df[col].apply(
                lambda x: self.normalizar_utf8(x) if pd.notna(x) else x
            )
        print(f"UTF-8 normalizado")

        # 2. Eliminar espacios
        print(f"\nEliminando espacios en blanco...")
        for col in text_cols:
            self.df[col] = self.df[col].str.strip()
        print(f"Espacios eliminados")

        # 3. Eliminar duplicados
        print(f"\nEliminando duplicados...")
        n_duplicados = self.df.duplicated().sum()
        self.df = self.df.drop_duplicates()
        print(f"{n_duplicados} duplicados eliminados")

        n_despues = len(self.df)
        print(f"\nEstado final: {n_despues:,} registros")
        print(f"Reducción: {n_antes - n_despues} registros ({((n_antes - n_despues) / n_antes * 100):.2f}%)")

        self.report['records_removed'] = n_antes - n_despues

    def validar_limpieza(self):
        """Validar integridad después de limpieza"""
        print(f"VALIDACIÓN POST-LIMPIEZA - {self.name.upper()}")

        print(f"\n✓ Registros: {len(self.df):,}")
        print(f"✓ Columnas: {len(self.df.columns)}")
        print(f"✓ Valores nulos: {self.df.isnull().sum().sum()}")
        print(f"✓ Duplicados: {self.df.duplicated().sum()}")
        print(f"✓ Memoria: {self.df.memory_usage(deep=True).sum() / 1024:.2f} KB")
        print(f"\nDataset limpio y validado")

    def generar_resumen(self):
        """Generar resumen final"""
        print(f"RESUMEN EJECUTIVO - {self.name.upper()}")

        return self.report

    def exportar_csv(self, filename):
        """Exportar CSV limpio"""
        self.df.to_csv(filename, index=False, encoding='utf-8')
        print(f"\nArchivo exportado: {filename}")

print("Clase DataScientistAnalyzer creada correctamente\n")


Clase DataScientistAnalyzer creada correctamente



In [ ]:
# SECCIÓN 3: ANÁLISIS DE CADA DATASET

analyzers = {}
reports = {}

for name, df in datasets.items():
    print(f"\n\n{'#'*90}")
    print(f"#{'  ANALIZANDO DATASET: ' + name.upper():^86}#")
    print(f"{'#'*90}")

    analyzer = DataScientistAnalyzer(name, df)

    # Ejecutar análisis completo
    analyzer.info_general()
    analyzer.mostrar_columnas()
    analyzer.estadisticas_descriptivas()
    analyzer.analizar_datos_faltantes()
    analyzer.detectar_outliers()
    analyzer.analizar_correlaciones()
    analyzer.analizar_distribuciones()
    analyzer.generar_wordcloud()

    # Limpiar datos
    analyzer.limpiar_datos()
    analyzer.validar_limpieza()

    # Generar resumen
    reports[name] = analyzer.generar_resumen()
    analyzers[name] = analyzer


Output hidden; open in https://colab.research.google.com to view.

In [ ]:
# SECCIÓN 4: COMPARATIVA DE LOS 3 DATASETS
# ============================================================================


print(f"#{'  COMPARATIVA DE LOS 3 DATASETS':^86}#")



print(f"TABLA COMPARATIVA GENERAL")

comparison_data = []
for name, analyzer in analyzers.items():
    comparison_data.append({
        'Dataset': name.upper(),
        'Registros': len(analyzer.df),
        'Columnas': len(analyzer.df.columns),
        'Valores Nulos': analyzer.df.isnull().sum().sum(),
        'Duplicados': analyzer.df.duplicated().sum(),
        'Memoria (KB)': analyzer.df.memory_usage(deep=True).sum() / 1024
    })

comparison_df = pd.DataFrame(comparison_data)
print(comparison_df.to_string(index=False))


#                             COMPARATIVA DE LOS 3 DATASETS                            #
TABLA COMPARATIVA GENERAL
                  Dataset  Registros  Columnas  Valores Nulos  Duplicados  Memoria (KB)
      TEST_30_PREDEFINIDO         30         7             21           0      9.164062
    OUTFITS_100K_COMPLETO     100000         6          54910           0  21769.640625
PRENDAS_BASE_500_MEJORADO        500         1              0           0    118.626953
